In [1]:
import os
import json
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime

# Configuration
LANGCHAIN_DIR = Path.cwd()
# Use JSON files for full database provenance (PMCID, text_element_id)
INPUT_JSON_DIR = LANGCHAIN_DIR / "test_results_50_docs" / "relevant_texts"
OUTPUT_DIR = LANGCHAIN_DIR / "summarization_results"
SUMMARIES_DIR = OUTPUT_DIR / "summaries"
RULES_DIR = OUTPUT_DIR / "rules"
AUDIT_DIR = OUTPUT_DIR / "audit_trails"

# Create output directories
for directory in [OUTPUT_DIR, SUMMARIES_DIR, RULES_DIR, AUDIT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Input JSON (with DB provenance): {INPUT_JSON_DIR}")
print(f"Output directory:                {OUTPUT_DIR}")
print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Input JSON (with DB provenance): /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/test_results_50_docs/relevant_texts
Output directory:                /Users/emir/Documents/GitHub/nlp-histo/langchain-summarization/summarization_results

Started at: 2026-01-26 20:01:13


In [2]:
# !pip install langchain langchain-openai langchain-core python-dotenv tiktoken

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

# Load environment variables (API keys)
load_dotenv()

# Verify API key is set
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️  Warning: OPENAI_API_KEY not found in environment")
    print("Please set it in .env file or export OPENAI_API_KEY=your_key")
else:
    print("✅ OpenAI API key loaded successfully")

/Users/emir/Documents/GitHub/nlp-histo/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ OpenAI API key loaded successfully


### Let's load all json files from the INPUT_JSON_DIR directory.

In [4]:
def load_json_files_with_provenance(input_dir: Path, limit: Optional[int] = None) -> List[Dict]:
    """
    Load UMLS entity JSON files with FULL DATABASE PROVENANCE.
    
    Each sentence includes:
    - pmcid: Links to documents.pmcid in database
    - text_element_id: Links to text_elements.id in database
    - section: Section context from text_elements.path_string
    - entity_text, start_char, end_char: Entity position
    - umls_score: Entity linking confidence
    
    This provides complete traceability back to the database schema.
    
    Args:
        input_dir: Directory containing JSON files
        limit: Optional limit on number of files to load
    
    Returns:
        List of dicts with full provenance metadata
    """
    json_files = sorted(input_dir.glob("*.json"))
    
    if limit:
        json_files = json_files[:limit]
    
    loaded_files = []
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata
            cui = data.get('umls_cui', '')
            concept_name = data.get('canonical_name', '')
            entity_label = data.get('entity_label', '')
            
            # Extract sentences with full provenance
            sentences_with_provenance = []
            for sent_data in data.get('sentences', []):
                sentences_with_provenance.append({
                    'pmcid': sent_data.get('pmcid'),  # Links to documents.pmcid
                    'text_element_id': sent_data.get('text_element_id'),  # Links to text_elements.id
                    'sentence': sent_data.get('sentence'),
                    'section': sent_data.get('section'),
                    'entity_text': sent_data.get('entity_text'),
                    'start_char': sent_data.get('start_char'),
                    'end_char': sent_data.get('end_char'),
                    'umls_score': sent_data.get('umls_score')
                })
            
            # Group by PMCID for statistics
            pmcids = set(s['pmcid'] for s in sentences_with_provenance if s['pmcid'])
            
            loaded_files.append({
                'cui': cui,
                'concept_name': concept_name,
                'entity_label': entity_label,
                'filename': json_file.name,
                'filepath': str(json_file),
                'sentences_with_provenance': sentences_with_provenance,
                # Simple sentence list for compatibility
                'sentences': [s['sentence'] for s in sentences_with_provenance],
                'num_sentences': len(sentences_with_provenance),
                'num_documents': len(pmcids),
                'pmcids': list(pmcids),
                'metadata': {
                    'umls_cui': cui,
                    'canonical_name': concept_name,
                    'entity_label': entity_label,
                    'total_occurrences': data.get('total_occurrences', 0),
                    'unique_entity_texts': data.get('unique_entity_texts', [])
                }
            })
        
        except Exception as e:
            print(f"Warning: Could not load {json_file.name}: {e}")
    
    return loaded_files


# Load JSON files with full provenance
print("Loading JSON files with database provenance...")
json_files = load_json_files_with_provenance(INPUT_JSON_DIR, limit=10)

print(f"\nLoaded {len(json_files)} concept files with DB provenance")
print("\nSample file structure:")
if json_files:
    sample = json_files[0]
    print(f"  CUI: {sample['cui']}")
    print(f"  Concept: {sample['concept_name']}")
    print(f"  Sentences: {sample['num_sentences']}")
    print(f"  Documents (PMCIDs): {sample['num_documents']}")
    if sample['sentences_with_provenance']:
        sent = sample['sentences_with_provenance'][0]
        print("\n  Sample sentence provenance:")
        print(f"    PMCID: {sent['pmcid']} -> documents.pmcid")
        print(f"    text_element_id: {sent['text_element_id']} -> text_elements.id")
        print(f"    section: {sent['section']}")
        print(f"    umls_score: {sent['umls_score']}")

Loading JSON files with database provenance...

Loaded 10 concept files with DB provenance

Sample file structure:
  CUI: C0000727
  Concept: Abdomen, Acute
  Sentences: 2
  Documents (PMCIDs): 2

  Sample sentence provenance:
    PMCID: PMC9355740 -> documents.pmcid
    text_element_id: 198 -> text_elements.id
    section: Severe autoimmune haemolytic anaemia following SARS-CoV-2 vaccination in patients with treatment naïve B-cell neoplasms: a case series
    umls_score: 0.8242956399917603


### Get the summary for each file.

In [1]:
# =============================================================================
# AUDITABLE MAP/REDUCE SYSTEM PROMPTS
# =============================================================================
# This system maintains full traceability from final summary -> chunks -> source sentences -> database
#
# Provenance Chain:
#   Final Summary [claim] -> Chunk ID -> Sentence IDs -> PMCID + text_element_id (database)
#
# Output Format: Structured JSON blocks enable machine parsing for audit validation
# =============================================================================

MAP_SYSTEM_PROMPT = """<Role>You are a High-Recall Medical Evidence Analyst specializing in histopathology.</Role>

<Task>
Convert the provided literature chunk into a list of ATOMIC, INDEPENDENT medical facts with full provenance.

CRITICAL EXTRACTION RULES:
1. ZERO LOSS: Extract every unique dose, p-value, patient count, demographic, and clinical relationship.
2. ATOMICITY: Each 'finding' must be a standalone fact. If a sentence/paragraph contains multiple observations (e.g., two different stains), you MUST create separate entries.
3. NO CONTEXTUAL DRIFT: Do not summarize. If information is partial due to chunk boundaries, extract exactly what is visible.
4. TELEGRAPHIC STYLE: Omit filler like 'the authors found.' Focus on direct relationships using arrows (e.g., 'CD30 -> Positive').
5. CITATION: Every claim MUST cite its Sentence ID using the format: [S1|PMC123456|789].
</Task>

<Categories>
Assign each finding to exactly one: morphology | IHC | molecular_genetics | staging | treatment | prognosis
</Categories>

<FilterRules>
SKIP: Author names, journal metadata, acknowledgments, funding, and generic boilerplate.
ONLY extract from: methods, results, discussion, and case descriptions.
</FilterRules>

<OutputFormat>
Return your analysis in this EXACT structure:

```json
{{
  "chunk_id": "{chunk_id}",
  "findings": [
    {{
      "category": "morphology|IHC|molecular_genetics|staging|treatment|prognosis",
      "claim": "<telegraphic_atomic_fact>",
      "evidence": ["S1|PMC123456|789"],
      "confidence": "high|medium|low",
      "verbatim_support": "<key_quote>"
    }}
  ],
  "summary_text": "<narrative_summary_with_inline_citations>",
  "audit_metadata": {{
    "sentences_analyzed": <count>,
    "sentences_cited": [<ids>],
    "pmcids_referenced": [<pmcids>],
    "uncited_sentences": [<ids>]
  }}
}}"""

MAP_USER_PROMPT = """<Context>
Concept: {concept_name}
Chunk ID: {chunk_id}
Source Sentences (each tagged with [SentenceID|PMCID|TextElementID]):
{text}
</Context>"""

In [ ]:
REDUCE_SYSTEM_PROMPT = """
<Role>
You are a Lead Pathologist synthesizing chunk-level analyses into a Master Clinical Brief with FULL AUDIT TRAIL.
</Role>

<Task>
Consolidate all chunk analyses into a unified report while PRESERVING THE COMPLETE AUDIT CHAIN.

CRITICAL AUDIT REQUIREMENTS:
1. EVERY claim must cite the source using format: [S1|PMC123456|789]
2. When multiple sources support a finding, list ALL citations
3. Flag any conflicting evidence with all sources cited
4. Do NOT add information not present in the chunk analyses
5. PRESERVE QUANTITATIVE NUANCE: Do not average or generalize specific values (percentages, doses, p-values). If findings differ, list them as a range or as distinct supporting points.
</Task>

<OutputFormat>
```json
{{
  "concept": "{concept_name}",
  "sections": {{
    "clinical_significance": {{
      "findings": [
        {{
          "claim": "<statement>",
          "sources": [
            {{"sentences": ["S1|PMC123456|789"], "verbatim": "<quote>"}}
          ],
          "strength": "strong|moderate|weak"
        }}
      ]
    }},
    "histopathological_features": {{ ... }},
    "management_outcomes": {{ ... }},
    "risk_factors_associations": {{ ... }}
  }},
  "narrative_summary": "<readable summary with inline citations [S1|PMC123456|789]>",
  "audit_trail": {{
    "chunks_processed": <count>,
    "total_sentences_cited": <count>,
    "unique_pmcids": [<list>],
    "unique_text_element_ids": [<list>],
    "evidence_conflicts": [
      {{"topic": "...", "conflicting_sources": [...]}}
    ]
  }}
}}
```
</OutputFormat>

Master Auditable Summary:"""

REDUCE_USER_PROMPT = """<Context>
Concept: {concept_name}
Total Chunks: {num_chunks}

Chunk Analyses (JSON format with provenance):
{summaries}
</Context>"""

In [7]:
RULE_EXTRACTION_SYSTEM_PROMPT = """
<Role>
You are a Medical Knowledge Engineer. Extract structured IF-THEN rules with COMPLETE PROVENANCE from the Auditable Summary.
</Role>

<Task>
Extract actionable clinical rules, each with FULL TRACEABILITY back to source documents.

AUDIT REQUIREMENTS:
1. Each rule must cite specific evidence from the summary
2. Include the database reference (PMCID + text_element_id) for each supporting sentence
3. Use the citation format: S1|PMC123456|789
</Task>

<OutputFormat>
```json
{{
  "rules": [
    {{
      "rule_id": "R1",
      "type": "Diagnostic|Prognostic|Management",
      "condition": "IF <observation>",
      "action": "THEN <conclusion>",
      "confidence": "High|Medium|Low",
      "evidence_chain": [
        {{
          "sentence_id": "S1",
          "pmcid": "PMC123456",
          "text_element_id": 789,
          "verbatim": "<supporting quote>"
        }}
      ],
      "contraindications": ["<any noted exceptions with sources>"]
    }}
  ],
  "audit_summary": {{
    "total_rules": <count>,
    "rules_by_type": {{"Diagnostic": N, "Prognostic": N, "Management": N}},
    "pmcids_supporting_rules": [<list>],
    "average_evidence_per_rule": <float>
  }}
}}
```
</OutputFormat>

Extracted Rules with Provenance:"""

RULE_EXTRACTION_USER_PROMPT = """<Input>
Concept: {concept_name}
Auditable Summary (JSON with full provenance):
{summary}
</Input>"""

#### Generate template from prompt for summarization:

In [8]:
MAPPING_TEMPLATE = ChatPromptTemplate(
    [('system', MAP_SYSTEM_PROMPT),
     ('user', MAP_USER_PROMPT)]
)

REDUCE_TEMPLATE = ChatPromptTemplate(
    [('system', REDUCE_SYSTEM_PROMPT),
     ('user', REDUCE_USER_PROMPT)]
)

RULE_EXTRACTION_TEMPLATE = ChatPromptTemplate(
    [('system', RULE_EXTRACTION_SYSTEM_PROMPT),
     ('user', RULE_EXTRACTION_USER_PROMPT)]
)

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Dict, Literal

# =============================================================================
# MAP SCHEMA - matches MAP_PROMPT output format
# =============================================================================
class Finding(BaseModel):
    category: Literal["morphology", "IHC", "molecular_genetics", "staging", "treatment", "prognosis"]
    claim: str = Field(description="The factual medical statement")
    evidence: List[str] = Field(description="List of citation IDs e.g. S1|PMC123|te456")
    confidence: Literal["high", "medium", "low"]
    verbatim_support: str = Field(description="Exact quote from the source text")

class AuditMetadata(BaseModel):
    sentences_analyzed: int
    sentences_cited: List[str]
    pmcids_referenced: List[str]
    uncited_sentences: List[str] = Field(description="List of sentence IDs not used in findings")

class AuditableSummary(BaseModel):
    chunk_id: str
    findings: List[Finding]
    summary_text: str
    audit_metadata: AuditMetadata


# =============================================================================
# REDUCE SCHEMA - matches REDUCE_PROMPT output format
# =============================================================================
class SourceReference(BaseModel):
    sentences: List[str] = Field(description="List of citation IDs e.g. S1|PMC123|te456")
    verbatim: str

class SectionFinding(BaseModel):
    claim: str
    sources: List[SourceReference]
    strength: Literal["strong", "moderate", "weak"]

class Section(BaseModel):
    findings: List[SectionFinding]

# NEW: Explicitly define the sections instead of using a Dict
class SectionsContainer(BaseModel):
    clinical_significance: Section
    histopathological_features: Section
    management_outcomes: Section
    risk_factors_associations: Section

class EvidenceConflict(BaseModel):
    topic: str
    conflicting_sources: List[str]

class ReduceAuditTrail(BaseModel):
    chunks_processed: int
    total_sentences_cited: int
    unique_pmcids: List[str]
    unique_text_element_ids: List[int]
    evidence_conflicts: List[EvidenceConflict] 

class ConsolidatedSummary(BaseModel):
    concept: str
    sections: SectionsContainer
    narrative_summary: str
    audit_trail: ReduceAuditTrail


# =============================================================================
# RULE EXTRACTION SCHEMA - matches RULE_EXTRACTION_PROMPT output format
# =============================================================================
class EvidenceChainItem(BaseModel):
    sentence_id: str
    pmcid: str
    text_element_id: int
    verbatim: str

class Rule(BaseModel):
    rule_id: str
    type: Literal["Diagnostic", "Prognostic", "Management"]
    condition: str = Field(description="IF <observation>")
    action: str = Field(description="THEN <conclusion>")
    confidence: Literal["High", "Medium", "Low"]
    evidence_chain: List[EvidenceChainItem]
    contraindications: List[str]

class RuleCounts(BaseModel):
    Diagnostic: int = Field(description="Count of Diagnostic rules")
    Prognostic: int = Field(description="Count of Prognostic rules")
    Management: int = Field(description="Count of Management rules")

class RuleAuditSummary(BaseModel):
    total_rules: int
    rules_by_type: RuleCounts
    pmcids_supporting_rules: List[str]
    average_evidence_per_rule: float

class ExtractedRules(BaseModel):
    rules: List[Rule]
    audit_summary: RuleAuditSummary


# =============================================================================
# LLM and Chain Setup
# =============================================================================
base_cheap_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)
base_smart_llm = ChatOpenAI(model="gpt-4o", temperature=0)

map_llm = base_cheap_llm.with_structured_output(
    AuditableSummary, strict=True
).with_retry(stop_after_attempt=3)

reduce_llm = base_smart_llm.with_structured_output(
    ConsolidatedSummary, strict=True
).with_retry(stop_after_attempt=3)

rule_llm = base_smart_llm.with_structured_output(
    ExtractedRules, strict=True
).with_retry(stop_after_attempt=3)

map_chain = MAPPING_TEMPLATE | map_llm
reduce_chain = REDUCE_TEMPLATE | reduce_llm
rule_chain = RULE_EXTRACTION_TEMPLATE | rule_llm

In [10]:
import tiktoken

class TokenTracker:
    def __init__(self):
        self.totals = {
            "gpt-4o-mini": {"input": 0, "output": 0},
            "gpt-4o": {"input": 0, "output": 0}
        }
        # Prices per 1M tokens (as of early 2026 - verify with latest docs)
        self.prices = {
            "gpt-4o-mini": {"input": 0.15, "output": 0.60},
            "gpt-4o": {"input": 2.50, "output": 10.00}
        }

    def add_usage(self, model: str, input_str: str, output_obj: any):
        enc = tiktoken.encoding_for_model(model)
        
        # Count input tokens
        self.totals[model]["input"] += len(enc.encode(input_str))
        
        # Count output tokens (convert Pydantic object back to string)
        output_str = json.dumps(output_obj.model_dump())
        self.totals[model]["output"] += len(enc.encode(output_str))

    def calculate_total_cost(self):
        total_cost = 0
        for model, usage in self.totals.items():
            in_cost = (usage["input"] / 1_000_000) * self.prices[model]["input"]
            out_cost = (usage["output"] / 1_000_000) * self.prices[model]["output"]
            total_cost += (in_cost + out_cost)
        return round(total_cost, 4)

In [ ]:
# =============================================================================
# PIPELINE CACHE - Avoid redundant LLM calls for identical inputs
# =============================================================================
# Keys are deterministic based on text_element_ids (no hash collisions possible)
# Cache persists to disk and survives kernel restarts
# =============================================================================

CACHE_FILE = OUTPUT_DIR / "pipeline_cache.json"

class PipelineCache:
    """
    Caches MAP, REDUCE, and RULE outputs using text_element_ids as keys.
    No hashing = no collision risk.
    """
    
    def __init__(self, cache_path: Path = CACHE_FILE):
        self.cache_path = cache_path
        self.map_cache: Dict[str, dict] = {}      # chunk_key -> AuditableSummary
        self.reduce_cache: Dict[str, dict] = {}   # reduce_key -> ConsolidatedSummary  
        self.rule_cache: Dict[str, dict] = {}     # rule_key -> ExtractedRules
        self.stats = {"map_hits": 0, "map_misses": 0, 
                      "reduce_hits": 0, "reduce_misses": 0,
                      "rule_hits": 0, "rule_misses": 0}
        self._load()
    
    def _load(self):
        """Load cache from disk if it exists."""
        if self.cache_path.exists():
            try:
                with open(self.cache_path, 'r') as f:
                    data = json.load(f)
                self.map_cache = data.get('map', {})
                self.reduce_cache = data.get('reduce', {})
                self.rule_cache = data.get('rule', {})
                print(f"Loaded cache: {len(self.map_cache)} map, {len(self.reduce_cache)} reduce, {len(self.rule_cache)} rule entries")
            except Exception as e:
                print(f"Warning: Could not load cache: {e}")
    
    def save(self):
        """Persist cache to disk."""
        with open(self.cache_path, 'w') as f:
            json.dump({
                'map': self.map_cache,
                'reduce': self.reduce_cache,
                'rule': self.rule_cache
            }, f)
    
    # -------------------------------------------------------------------------
    # Key Generation (deterministic, collision-free)
    # -------------------------------------------------------------------------
    
    @staticmethod
    def get_chunk_key(sentences: list) -> str:
        """Key from sorted text_element_ids - unique by definition."""
        ids = sorted(s.get('text_element_id', 0) for s in sentences)
        return ",".join(map(str, ids))
    
    @staticmethod
    def get_reduce_key(summaries: list, concept_name: str) -> str:
        """Key from chunk_ids of input summaries + concept."""
        # summaries are either AuditableSummary or ConsolidatedSummary objects/dicts
        chunk_ids = []
        for s in summaries:
            if hasattr(s, 'chunk_id'):
                chunk_ids.append(s.chunk_id)
            elif hasattr(s, 'concept'):
                # ConsolidatedSummary - use its audit trail
                chunk_ids.append(f"consolidated:{s.audit_trail.chunks_processed}")
            elif isinstance(s, dict):
                chunk_ids.append(s.get('chunk_id', s.get('concept', 'unknown')))
        return f"{concept_name}|" + ";".join(sorted(chunk_ids))
    
    @staticmethod
    def get_rule_key(summary: 'ConsolidatedSummary', concept_name: str) -> str:
        """Key from the consolidated summary's unique identifiers."""
        if hasattr(summary, 'audit_trail'):
            te_ids = sorted(summary.audit_trail.unique_text_element_ids)
        elif isinstance(summary, dict):
            te_ids = sorted(summary.get('audit_trail', {}).get('unique_text_element_ids', []))
        else:
            te_ids = []
        return f"{concept_name}|rule|" + ",".join(map(str, te_ids))
    
    # -------------------------------------------------------------------------
    # Cache Operations
    # -------------------------------------------------------------------------
    
    def get_map(self, sentences: list) -> Optional[AuditableSummary]:
        """Get cached MAP result or None."""
        key = self.get_chunk_key(sentences)
        if key in self.map_cache:
            self.stats["map_hits"] += 1
            return AuditableSummary(**self.map_cache[key])
        self.stats["map_misses"] += 1
        return None
    
    def set_map(self, sentences: list, result: AuditableSummary):
        """Cache a MAP result."""
        key = self.get_chunk_key(sentences)
        self.map_cache[key] = result.model_dump()
    
    def get_reduce(self, summaries: list, concept_name: str) -> Optional[ConsolidatedSummary]:
        """Get cached REDUCE result or None."""
        key = self.get_reduce_key(summaries, concept_name)
        if key in self.reduce_cache:
            self.stats["reduce_hits"] += 1
            return ConsolidatedSummary(**self.reduce_cache[key])
        self.stats["reduce_misses"] += 1
        return None
    
    def set_reduce(self, summaries: list, concept_name: str, result: ConsolidatedSummary):
        """Cache a REDUCE result."""
        key = self.get_reduce_key(summaries, concept_name)
        self.reduce_cache[key] = result.model_dump()
    
    def get_rule(self, summary, concept_name: str) -> Optional[ExtractedRules]:
        """Get cached RULE result or None."""
        key = self.get_rule_key(summary, concept_name)
        if key in self.rule_cache:
            self.stats["rule_hits"] += 1
            return ExtractedRules(**self.rule_cache[key])
        self.stats["rule_misses"] += 1
        return None
    
    def set_rule(self, summary, concept_name: str, result: ExtractedRules):
        """Cache a RULE result."""
        key = self.get_rule_key(summary, concept_name)
        self.rule_cache[key] = result.model_dump()
    
    def print_stats(self):
        """Print cache hit/miss statistics."""
        print("\nCache Statistics:")
        print(f"  MAP:    {self.stats['map_hits']} hits / {self.stats['map_misses']} misses")
        print(f"  REDUCE: {self.stats['reduce_hits']} hits / {self.stats['reduce_misses']} misses")
        print(f"  RULE:   {self.stats['rule_hits']} hits / {self.stats['rule_misses']} misses")
        total_hits = self.stats['map_hits'] + self.stats['reduce_hits'] + self.stats['rule_hits']
        total = total_hits + self.stats['map_misses'] + self.stats['reduce_misses'] + self.stats['rule_misses']
        if total > 0:
            print(f"  Overall hit rate: {total_hits/total:.1%}")


# Initialize global cache
pipeline_cache = PipelineCache()
print(f"Cache file: {CACHE_FILE}")

In [ ]:
def minify_summaries(summaries: List[BaseModel]) -> str:
    """
    Converts Pydantic objects to a compact JSON string to save tokens.
    Removes all whitespace and newlines.
    """
    # model_dump(exclude_none=True) prevents sending empty fields
    data = [s.model_dump(exclude_none=True) for s in summaries]
    return json.dumps(data, separators=(',', ':'))

In [12]:
def format_sentences_for_llm(chunk: list) -> str:
    """
    Formats a list of sentence dictionaries into a structured string 
    for the LLM to easily reference citation IDs.
    """
    formatted_lines = []
    
    for i, item in enumerate(chunk):
        pmcid = item.get('pmcid', 'UNKNOWN')
        te_id = item.get('text_element_id', '0')
        sentence_text = item.get('sentence', '').strip()
        
        # Create a unique Citation ID for this specific sentence
        # Format: S{index}|{PMCID}|{ElementID}
        citation_id = f"S{i+1}|{pmcid}|{te_id}"
        
        # Format the line for the prompt
        formatted_lines.append(f"[{citation_id}] {sentence_text}")
    
    return "\n".join(formatted_lines)

In [ ]:

def process_document_hierarchical(file_data: Dict, map_chain, reduce_chain, rule_chain, 
                                   cache: PipelineCache = None) -> Dict:
    """
    Process document using hierarchical Map-Reduce with caching.
    
    Cache behavior:
    - MAP: Cached by text_element_ids in each chunk
    - REDUCE: Cached by chunk_ids of input summaries
    - RULE: Cached by text_element_ids in the consolidated summary
    
    Returns Pydantic objects with full provenance tracking.
    """
    concept_name = file_data['concept_name']
    cui = file_data['cui']
    sentences_with_metadata = file_data['sentences_with_provenance']
    
    try:
        # 2. MAP STEP: Process sentences in chunks of 10
        chunk_size = 10
        sentence_chunks = [sentences_with_metadata[i:i + chunk_size] 
                          for i in range(0, len(sentences_with_metadata), chunk_size)]
        
        # Check cache for each chunk, only call LLM for uncached chunks
        current_summaries: List[AuditableSummary] = []
        uncached_chunks = []
        uncached_indices = []
        
        for i, chunk in enumerate(sentence_chunks):
            if cache:
                cached_result = cache.get_map(chunk)
                if cached_result:
                    current_summaries.append(cached_result)
                    continue
            
            # Not cached - queue for LLM call
            uncached_chunks.append(chunk)
            uncached_indices.append(i)
            current_summaries.append(None)  # Placeholder
        
        # Batch call LLM for uncached chunks only
        if uncached_chunks:
            map_inputs = [
                {
                    "concept_name": concept_name, 
                    "chunk_id": f"C{uncached_indices[j]+1}", 
                    "text": format_sentences_for_llm(chunk)
                }
                for j, chunk in enumerate(uncached_chunks)
            ]
            new_results: List[AuditableSummary] = map_chain.batch(map_inputs)
            
            # Fill in placeholders and cache new results
            for j, (chunk, result) in enumerate(zip(uncached_chunks, new_results)):
                idx = uncached_indices[j]
                current_summaries[idx] = result
                if cache:
                    cache.set_map(chunk, result)
        
        # 3. REDUCE STEP: Recursively collapse summaries (with caching)
        if len(current_summaries) == 1:
            # Check cache first
            if cache:
                cached_reduce = cache.get_reduce(current_summaries, concept_name)
                if cached_reduce:
                    master_summary = cached_reduce
                else:
                    master_summary: ConsolidatedSummary = reduce_chain.invoke({
                        "concept_name": concept_name,
                        "num_chunks": 1,
                        "summaries": minify_summaries(current_summaries)
                    })
                    cache.set_reduce(current_summaries, concept_name, master_summary)
            else:
                master_summary: ConsolidatedSummary = reduce_chain.invoke({
                    "concept_name": concept_name,
                    "num_chunks": 1,
                    "summaries": minify_summaries(current_summaries)
                })
        else:
            while len(current_summaries) > 1:
                summary_groups = [current_summaries[i:i + 10] 
                                 for i in range(0, len(current_summaries), 10)]
                
                next_level_summaries = []
                uncached_groups = []
                uncached_group_indices = []
                
                # Check cache for each reduce group
                for i, group in enumerate(summary_groups):
                    if cache:
                        cached_reduce = cache.get_reduce(group, concept_name)
                        if cached_reduce:
                            next_level_summaries.append(cached_reduce)
                            continue
                    
                    uncached_groups.append(group)
                    uncached_group_indices.append(i)
                    next_level_summaries.append(None)  # Placeholder
                
                # Batch call LLM for uncached groups
                if uncached_groups:
                    reduce_inputs = [
                        {
                            "concept_name": concept_name,
                            "num_chunks": len(group),
                            "summaries": minify_summaries(group)
                        }
                        for group in uncached_groups
                    ]
                    new_reduces: List[ConsolidatedSummary] = reduce_chain.batch(reduce_inputs)
                    
                    for j, (group, result) in enumerate(zip(uncached_groups, new_reduces)):
                        idx = uncached_group_indices[j]
                        next_level_summaries[idx] = result
                        if cache:
                            cache.set_reduce(group, concept_name, result)
                
                current_summaries = next_level_summaries

            master_summary: ConsolidatedSummary = current_summaries[0]
        
        # 4. RULE EXTRACTION: Run on the final Master Summary (with caching)
        if cache:
            cached_rules = cache.get_rule(master_summary, concept_name)
            if cached_rules:
                final_rules = cached_rules
            else:
                final_rules: ExtractedRules = rule_chain.invoke({
                    "concept_name": concept_name,
                    "summary": minify_summaries([master_summary])
                })
                cache.set_rule(master_summary, concept_name, final_rules)
        else:
            final_rules: ExtractedRules = rule_chain.invoke({
                "concept_name": concept_name,
                "summary": minify_summaries([master_summary])
            })
        
        return {
            'status': 'success',
            'cui': cui,
            'concept_name': concept_name,
            'summary': master_summary.narrative_summary,
            'rules': [rule.model_dump() for rule in final_rules.rules],
            'audit_trail': {
                'master_summary': master_summary.model_dump(),
                'rules_provenance': final_rules.model_dump()
            }
        }

    except Exception as e:
        return {
            'status': 'error',
            'cui': cui,
            'concept_name': concept_name,
            'error': str(e)
        }

## Start processing.

In [ ]:
def get_result_path(cui: str) -> Path:
    """Get the path where a result would be saved."""
    return SUMMARIES_DIR / f"{cui}.json"

def load_existing_result(cui: str) -> Optional[Dict]:
    """Load an existing result if it exists."""
    result_path = get_result_path(cui)
    if result_path.exists():
        with open(result_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def save_result(result: Dict) -> None:
    """Save a result to disk."""
    cui = result['cui']
    result_path = get_result_path(cui)
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)


results = []
errors = []
skipped = []

print("="*80)
print("Starting Processing Pipeline (with caching)")
print("="*80 + "\n")

for i, file_data in enumerate(json_files, 1):
    cui = file_data['cui']
    concept_name = file_data['concept_name']
    
    print(f"[{i}/{len(json_files)}] {cui} - {concept_name[:50]}...", end=" ")
    
    # Check if already processed (final result cache)
    existing = load_existing_result(cui)
    if existing is not None:
        print("SKIPPED (cached)")
        skipped.append(existing)
        continue
    
    # Process with intermediate caching
    result = process_document_hierarchical(
        file_data, 
        map_chain=map_chain, 
        reduce_chain=reduce_chain, 
        rule_chain=rule_chain,
        cache=pipeline_cache  # Pass the cache
    )
    
    if result['status'] == 'success':
        save_result(result)
        results.append(result)
        print(f"OK ({len(result['summary'])} chars, {len(result['rules'])} rules)")
    else:
        errors.append(result)
        print(f"ERROR: {result['error']}")

# Save the cache to disk after processing
pipeline_cache.save()
pipeline_cache.print_stats()

print("\n" + "="*80)
print("Processing Complete:")
print(f"  New:    {len(results)}")
print(f"  Cached: {len(skipped)}")
print(f"  Errors: {len(errors)}")
print("="*80)